# Chapter 2: Attention from scratch

Tokenization turned our text into vectors. But those vectors sit there in isolation, each token knowing nothing about the ones around it. Language does not work like that: the meaning of a word depends heavily on its neighbours. In *"the bank of the river"* versus *"money in the bank"*, the word **bank** only makes sense once we look at the other words.

**Attention** is the mechanism that fixes this. For every token, it asks a simple question: *"out of all the tokens in this sentence, which ones should I pay attention to, and how much?"* The answer is used to blend information from the whole sequence into each token.

In this chapter we build attention up one layer at a time:

- raw attention scores with a plain dot product
- turning those scores into weights with softmax, then into a context vector
- doing it all at once with matrix multiplication
- adding trainable weights (query, key, value)
- scaling, then causal masking so the model cannot peek at the future
- wrapping it in a reusable `CausalAttention` class
- stacking several of them into multi-head attention

## Step 1: Raw attention scores

We start with three tiny tokens, each a 3-dimensional vector. We pick the first token as our **query** and measure how similar it is to every token (including itself) using a **dot product**. The dot product is a cheap, natural measure of similarity: the more two vectors point in the same direction, the larger it is.

The result is one raw score per token, saying how relevant each one is to our query.

In [43]:
import torch 

inputs= torch.tensor([   #inputs for three words with 3 dimension
    [0.44, 0.15, 0.89],
    [0.55, 0.87, 0.66],
    [0.57, 0.85, 0.08]
])
print("inputs shape:", inputs.shape)
query= inputs[0]
print("query:", query)
attn_scores_2= torch.empty(inputs.shape[0])
for i,x_i in enumerate(inputs):
    attn_scores_2[i]= torch.dot(query, x_i)
    
print("raw attention scores:", attn_scores_2)

inputs shape: torch.Size([3, 3])
query: tensor([0.4400, 0.1500, 0.8900])
raw attention scores: tensor([1.0082, 0.9599, 0.4495])


In [44]:
import torch 
attn_scores_2= torch.tensor([1.0082, 0.9599, 0.4495])
attn_weights_2= torch.softmax(attn_scores_2, dim=0)
inputs= torch.tensor([
    [0.43, 0.15, 0.89],
    [0.55, 0.87, 0.66],
    [0.57, 0.85, 0.08]
])
context_vec_2= torch.zeros(inputs.shape[1])
for i,x_i in enumerate(inputs):
    context_vec_2+= attn_weights_2[i]*x_i
print("attention weights(percentages):", attn_weights_2)
print("final context vector:", context_vec_2)

attention weights(percentages): tensor([0.3961, 0.3774, 0.2265])
final context vector: tensor([0.5070, 0.5803, 0.6197])


## Doing it all at once with matrix multiplication

The loop above worked, but it only handled a single query and used an explicit `for` loop. Both of those are slow and clumsy once sequences get long.

The whole operation is really just matrix multiplication in disguise. If we multiply the input matrix by its own transpose, we get **every** pairwise score in one shot: row *i*, column *j* is the dot product of token *i* with token *j*.

From there:

- `softmax` along each row turns scores into weights that sum to 1
- multiplying those weights by the inputs gives every context vector at once

No loops, no single-query restriction. This is the vectorised form we will build on from here.

In [45]:
attn_scores= torch.matmul(inputs, inputs.T)
attn_weights = torch.softmax(attn_scores, dim=1)
all_context_vec= torch.matmul(attn_weights,inputs)
print("all raw scores matrix:\n", attn_scores)
print("\nall attention weights matrix(row-wise sum=1):\n", attn_weights)
print("\nall final context vectors matrix:\n", all_context_vec)

all raw scores matrix:
 tensor([[0.9995, 0.9544, 0.4438],
        [0.9544, 1.4950, 1.1058],
        [0.4438, 1.1058, 1.0538]])

all attention weights matrix(row-wise sum=1):
 tensor([[0.3953, 0.3779, 0.2268],
        [0.2577, 0.4425, 0.2998],
        [0.2092, 0.4057, 0.3851]])

all final context vectors matrix:
 tensor([[0.5071, 0.5808, 0.6194],
        [0.5251, 0.6785, 0.5454],
        [0.5326, 0.7116, 0.4848]])


## Self-attention with trainable weights (query, key, value)

So far attention had nothing to learn. Tokens were compared directly to each other, which gives a fixed, hard-wired notion of similarity. A real model needs to *learn* what "relevant" means for the task.

We do this by projecting every input into three different spaces using three weight matrices:

- **Query ($W_q$)** — what this token is looking for
- **Key ($W_k$)** — what this token offers to others
- **Value ($W_v$)** — the actual information this token passes along

Attention scores now come from comparing **queries against keys** (instead of raw inputs), and the final blend is taken over **values**. Because $W_q$, $W_k$ and $W_v$ are learnable, the model can shape its own idea of relevance during training.

> Here we keep `requires_grad=False` just to inspect the shapes cleanly; in a real model these would be trainable.

In [46]:
import torch 
x=inputs
d_in = x.shape[1]
d_out= 2
W_query= torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key= torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value= torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

keys= torch.matmul(x,W_key)
queries= torch.matmul(x, W_query)
values = torch.matmul(x, W_value)

print("queries matrix shape:", queries.shape)
print("keys matrix shape:", keys.shape)
print("values matrix shape:", values.shape)

queries matrix shape: torch.Size([3, 2])
keys matrix shape: torch.Size([3, 2])
values matrix shape: torch.Size([3, 2])


## Scaled dot-product attention

We now have queries, keys and values, so we can compute the real attention. Query-key dot products give the scores, softmax turns them into weights, and multiplying by the values produces the context vectors.

There is one important extra step: **scaling**. As the key dimension $d_k$ grows, the dot products grow large too. Feed very large numbers into softmax and it becomes extremely "peaky" — almost all the weight lands on a single token, and the gradients shrink toward zero.

To keep things stable we divide the scores by $\sqrt{d_k}$ before the softmax. This is exactly the *"scaled dot-product attention"* from the original Transformer paper.

In [47]:
attn_scores= torch.matmul(queries, keys.T)
d_k= keys.shape[-1]
scaled_attn_scores= attn_scores/(d_k**0.5)
attn_weights= torch.softmax(scaled_attn_scores, dim=1)
context_vectors=torch.matmul(attn_weights, values)
print("scaled attention weights matrix:\n", attn_weights)
print("\n final trainable context vector matrix shape:", context_vectors.shape)

scaled attention weights matrix:
 tensor([[0.3657, 0.3702, 0.2641],
        [0.3757, 0.3828, 0.2415],
        [0.3636, 0.3698, 0.2666]])

 final trainable context vector matrix shape: torch.Size([3, 2])


## Causal attention: why GPT hides the future

The attention we have built so far lets every token look at every other token — including the ones that come **after** it. That is fine for tasks where the whole sentence is available at once, but it breaks a GPT-style model.

GPT is trained to **predict the next word**. If a token were allowed to peek at future tokens, it would simply read off the answer instead of learning to predict it. That is cheating, and the model would be useless at generation time.

**Causal masking** prevents this. Before the softmax, we set every "future" position to $-\infty$. After softmax those entries become exactly 0, so each token can only attend to itself and the tokens before it.

You can see it in the result: the first row attends only to itself, the second row to the first two tokens, and so on — a telltale lower-triangular pattern.

In [48]:
context_length= attn_weights.shape[0]
mask= torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked_attn_scores= scaled_attn_scores.masked_fill(mask.bool(), -float('inf'))
causal_attn_weights= torch.softmax(masked_attn_scores, dim=-1)
print("causal (GPT-style) attention weights matrix:\n", causal_attn_weights)

causal (GPT-style) attention weights matrix:
 tensor([[1.0000, 0.0000, 0.0000],
        [0.4954, 0.5046, 0.0000],
        [0.3636, 0.3698, 0.2666]])


## A reusable `CausalAttention` class

We have all the pieces; now we package them into a clean, reusable `nn.Module` so we can drop attention into a larger model later.

A few things worth noting in this class:

- **`nn.Linear`** replaces the manual weight matrices — tidier, and trainable by default.
- The causal mask is stored with **`register_buffer`**, so it moves with the model (CPU/GPU) and is saved with it, but is *not* treated as a learnable parameter.
- **`forward`** works on batched 3-D input of shape `(batch, num_tokens, d_in)`, which is what real training data looks like.
- A **`dropout`** layer is applied to the attention weights for regularisation.

This is the real, production-shaped version of everything we derived by hand above.

In [ ]:
import torch
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out= d_out

        self.W_query=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value=nn.Linear(d_in, d_out, bias=qkv_bias)

        self.dropout= nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length),diagonal=1))

    def forward(self,x):
        b, num_tokens, d_in= x.shape

        queries= self.W_query(x)
        keys= self.W_key(x)
        values= self.W_value(x)

        attn_scores= torch.matmul(queries, keys.transpose(1,2))
        attn_scores.masked_fill_(self.mask[:num_tokens, :num_tokens].bool(), -float('inf'))
        attn_weights= torch.softmax(attn_scores/(keys.shape[-1]**0.5), dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec= torch.matmul(attn_weights, values)
        return context_vec

## Testing the class on a real 3-D batch

Before trusting a layer, check its shapes. We feed in a dummy batch of 8 sequences, each 4 tokens long, each token a 256-dimensional vector — shape `(8, 4, 256)`.

A correct attention layer should return a context vector for every token while keeping the batch and sequence dimensions intact, so we expect the output shape to match the input: `(8, 4, 256)`.

In [50]:
batch_inputs= torch.randn(8,4,256)

causal_layer=CausalAttention(d_in=256, d_out=256, context_length=4, dropout=0.0)
output_vectors= causal_layer(batch_inputs)
print("input batch shape:", batch_inputs.shape)
print("output batch shape:", output_vectors.shape)

input batch shape: torch.Size([8, 4, 256])
output batch shape: torch.Size([8, 4, 256])


## From one head to many

A single attention layer learns **one** way of relating tokens. But language has many kinds of relationships at once — grammar, subject-verb links, long-range references, and more. Expecting one head to capture all of them is asking a lot.

**Multi-head attention** runs several attention layers ("heads") in parallel, each with its own query/key/value projections, so each head can specialise in a different pattern. Their outputs are then concatenated and mixed back together.

## Implementing `MultiHeadAttentionWrapper`

The simplest way to build multi-head attention is to stack several of our `CausalAttention` layers side by side.

The idea:

- create `num_heads` independent heads, each producing `d_out // num_heads` dimensions, so together they add back up to `d_out`
- run the input through every head and **concatenate** their outputs along the last dimension
- pass the result through a final **output projection** (`nn.Linear`) that lets the heads' information mix

The heads are held in an `nn.ModuleList` so PyTorch registers each one's parameters correctly.

In [53]:
import torch
import torch.nn as nn

class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads= nn.ModuleList([CausalAttention(d_in, d_out//num_heads, context_length, dropout, qkv_bias)
        for _ in range(num_heads)])

        self.out_projection=nn.Linear(d_out, d_out)
    def forward(self,x):
        head_outputs=[head(x) for head in self.heads]
        context_vec= torch.cat(head_outputs, dim=-1)

        return self.out_projection(context_vec)



## Testing the multi-head dimensions

Same sanity check as before. We send in a batch of shape `(8, 4, 256)` and split the attention across 4 heads.

Each head quietly works in a 64-dimensional slice (`256 / 4`), and once the heads are concatenated and projected, the output should come back out as `(8, 4, 256)` — same shape in, same shape out, which means multi-head attention can be stacked just like any other layer.

In [54]:
batch_inputs= torch.randn(8,4,256)
mha_layer = MultiHeadAttentionWrapper(d_in=256, d_out=256, context_length=4, dropout=0.0, num_heads=4)
mha_outputs= mha_layer(batch_inputs)
print("input batch shape:", batch_inputs.shape)
print("multi-head output shape:", mha_outputs.shape)

input batch shape: torch.Size([8, 4, 256])
multi-head output shape: torch.Size([8, 4, 256])
